In [1]:
import numpy as np

# 定义一个自定义的数组容器

注意数组的容器并不一定要继承`ndarray`,只需要添加`__array__`方法即可,所以首先定义一个类:

In [13]:
class PEArray:
  def __init__(self, height, width, spad_size):
    self._h = height
    self._w = width
    self._spad = spad_size
    self._pe = np.random.rand(self._h, self._w)

  def __repr__(self):
    return f"{self.__class__.__name__}(h={self._h}, w={self._w})"

  def __array__(self, dtype=None):
    return self._pe


我们可以方便的初始化他,并且将其中的数据通过`np.array\np.asarray`方法进行获取.

In [14]:
pe = PEArray(3, 4, 8)
pe

PEArray(h=3, w=4)

In [15]:
np.asarray(pe)

array([[0.76335779, 0.69525953, 0.38476074, 0.17466267],
       [0.97378041, 0.56689024, 0.35271186, 0.55738085],
       [0.88528812, 0.35084729, 0.92067979, 0.7784478 ]])

`__array__`类似c++中的数据类型重载转换重载,所以我们可以传入一个`PEArray`对象到`numpy`的计算函数中进行计算,但是需要注意的是返回值类型肯定是`ndarray`了:

In [16]:
np.multiply(pe, 2)

array([[1.52671557, 1.39051905, 0.76952148, 0.34932535],
       [1.94756082, 1.13378048, 0.70542373, 1.11476169],
       [1.77057623, 0.70169459, 1.84135959, 1.55689561]])

那么如果我们既想使用`numpy`提供的方法,又想保持我们的数据类型不变,仅对类中的数据进行操作,那么需要通过`__array_ufunc__`和`__array_function__`进行适配. 首先从`__array_ufunc__`方法开始:

## `__array_ufunc__`

`__array_ufunc__`是一个`unary`操作函数的一个接口,即调用[ufunc](https://numpy.org/doc/stable/reference/ufuncs.html#ufuncs)是对数组元素进行`elemwise`的操作,比如`add\subtract\multiply\log\sin`等等.

每个`__array_ufunc__`接收参数如下:
-   `ufunc`, `ufunc`函数对象,比如numpy.xxx

-   `method`, 方法名,因为每个`ufunc`函数对象都有四个方法,所以还得选方法

-   `inputs`, 输入对象

-   `kwargs`, `ufunc`的可选参数

对于每个`ufunc`都有相同的输入参数、属性,这个可以去文档中看,主要是每个函数还对应了4个`method`:
|name | description|
|-|-|
|ufunc.reduce(array[, axis, dtype, out, …])|Reduces array’s dimension by one, by applying ufunc along one axis.|
|ufunc.accumulate(array[, axis, dtype, out])|Accumulate the result of applying the operator to all elements.|
|ufunc.reduceat(array, indices[, axis, …])|Performs a (local) reduce with specified slices over a single axis.|
|ufunc.outer(A, B, /, **kwargs)|Apply the ufunc op to all pairs (a, b) with a in A and b in B.|
|ufunc.at(a, indices[, b])|Performs unbuffered in place operation on operand ‘a’ for elements specified by ‘indices’.  |

接下来我们适配一个`__call__`方法,也就是直接调用的方法:

In [35]:
from numbers import Number


class PEArray:
  def __init__(self, height, width, spad_size, pe=None):
    self._h = height
    self._w = width
    self._spad = spad_size
    if pe is not None:
      self._pe = pe
    else:
      self._pe = np.random.rand(self._h, self._w)

  def __repr__(self):
    return f"{self.__class__.__name__}(h={self._h}, w={self._w})"

  def __array__(self, dtype=None):
    return self._pe

  def __array_ufunc__(self, ufunc, method, *inputs, **kwargs):
    if method == '__call__':
      scalars = []
      objects = []
      for input in inputs:
        if isinstance(input, Number):
          scalars.append(input)
        elif isinstance(input, self.__class__):
          if input._pe.shape != self._pe.shape:
            raise ValueError("inconsistent shape")
          objects.append(input._pe)
        else:
          return NotImplementedError("not support the other type")
      return self.__class__(self._h, self._w, self._spad, ufunc(*objects, *scalars, **kwargs))
    else:
      return NotImplementedError("now only support __call__!")


在编写以上代码时需要注意类内的`array`也会被传入到input里面的,所以不要手动再传入`self._pe`了. 还有就是要给自己类写一个合适的构造函数,以便于直接传入数组重新构造,接下来可以看到可以输出的正确的对象了.

In [39]:
a = PEArray(3,4,5)
b = 3.
c = PEArray(3,4,6)
print(np.add(a,b))
print(np.multiply(a,c))

PEArray(h=3, w=4)
PEArray(h=3, w=4)


同时对于一些方法也会正确处理,比如`np.sum`他默认是调用的`reduce`方法,此时我们并不能支持.

In [37]:
np.sum(a)

NotImplementedError('now only support __call__!')

但是还有个问题,我们此时没有继承python内部的操作符号:

In [45]:
a + b

TypeError: unsupported operand type(s) for +: 'PEArray' and 'float'

如果一个个继承比较麻烦,我们可以继承numpy内置的脚手架类`numpy.lib.mixins.NDArrayOperatorsMixin`

In [46]:
from numpy.lib.mixins import NDArrayOperatorsMixin


class PEArray(NDArrayOperatorsMixin):
  def __init__(self, height, width, spad_size, pe=None):
    self._h = height
    self._w = width
    self._spad = spad_size
    if pe is not None:
      self._pe = pe
    else:
      self._pe = np.random.rand(self._h, self._w)

  def __repr__(self):
    return f"{self.__class__.__name__}(h={self._h}, w={self._w})"

  def __array__(self, dtype=None):
    return self._pe

  def __array_ufunc__(self, ufunc, method, *inputs, **kwargs):
    if method == '__call__':
      scalars = []
      objects = []
      for input in inputs:
        if isinstance(input, Number):
          scalars.append(input)
        elif isinstance(input, self.__class__):
          if input._pe.shape != self._pe.shape:
            raise ValueError("inconsistent shape")
          objects.append(input._pe)
        else:
          return NotImplementedError("not support the other type")
      return self.__class__(self._h, self._w, self._spad, ufunc(*objects, *scalars, **kwargs))
    else:
      return NotImplementedError("now only support __call__!")


In [47]:
a = PEArray(1,2,3)
b = 10
a + b

PEArray(h=1, w=2)